# Primer trim — optional legacy notebook

Current Stage A decision is **not to trim VH/JH primers** before reconstruction, because multiplex primer trimming can remove V/J-informative bases. Keep this notebook only for controlled sensitivity tests.

In [ ]:
import os, sys, sysconfig
_CONDA_ENV = "/opt/conda/envs/bcr_env"
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [_CONDA_ENV + "/lib/python3.11/site-packages", sysconfig.get_path("purelib")]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ["HOME"] = "/data/user/epishkin"
os.environ["XDG_CONFIG_HOME"] = "/data/user/epishkin/.config"
os.makedirs(os.environ["XDG_CONFIG_HOME"], exist_ok=True)


In [ ]:
import subprocess
from pathlib import Path

PRIMERS = {
    "PRJEB40348": "primer_refs/human_primers.fasta",
    "PRJNA848968": "primer_refs/horse_primers.fasta",
}

def run_primer_trim(volume, dataset):
    if dataset not in PRIMERS:
        print(f"[primer_trim] {dataset}: no primer trimming configured; skip")
        return
    vol = Path(volume)
    src = vol / "results" / dataset / "trimmed" / "fastq"
    dst = vol / "results" / dataset / "pr_trimmed" / "fastq"
    dst.mkdir(parents=True, exist_ok=True)
    primers = vol / PRIMERS[dataset]
    if not primers.exists():
        raise FileNotFoundError(f"Primer FASTA not found: {primers}")
    for f in sorted(src.glob("*.fastq.gz")):
        out = dst / f.name.replace(".fastq.gz", ".primer.fastq")
        cmd = ["MaskPrimers.py", "score", "-s", str(f), "-p", str(primers), "--mode", "cut", "--outdir", str(dst), "--outname", out.stem]
        print("[primer_trim]", " ".join(cmd))
        subprocess.run(cmd, check=True)


### Disabled by default

Uncomment only for a sensitivity analysis.

In [ ]:
# run_primer_trim('/data/user/epishkin', 'PRJEB40348')
# run_primer_trim('/data/user/epishkin', 'PRJNA848968')
